In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import * 




In [0]:
%run /Workspace/fmcg_domain/databricks_fmcg_data_engineering/Setup/utilities

In [0]:
silver_schema,bronze_schema,gold_schema

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","orders","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

# bronze table data 

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"



In [0]:
silver_df = spark.sql(f"select * from {bronze_table}")


In [0]:
silver_df = silver_df.dropDuplicates(
    subset = [
        "order_id"
    ]
)

In [0]:
silver_df.select("order_id").distinct().count()



In [0]:
silver_df = silver_df.withColumn(
    "order_placement_date",
    coalesce(
        expr("try_to_date(order_placement_date, 'yyyy-MM-dd')"),
        expr("try_to_date(order_placement_date, 'yyyy/MM/dd')"),
        expr("try_to_date(order_placement_date, 'dd/MM/yyyy')"),
        expr("try_to_date(order_placement_date, 'dd-MM-yyyy')"),
        expr("try_to_date(regexp_replace(order_placement_date, '^[A-Za-z]+, ', ''), 'MMMM dd, yyyy')")
    )
)

In [0]:
display(silver_df)

In [0]:
products = spark.sql("select product_id,product_code from fmcg.silver.products;")

In [0]:
silver_df = silver_df.withColumn(
    "product_id",
    col(
        "product_id"
    ).cast(
        "string")
)

In [0]:
invalid_orders = silver_df.join(
    products,
    on="product_id",
    how="left_anti"
)

display(invalid_orders)

In [0]:
customers_dim = spark.sql("select customer_code as customer_id from fmcg.gold.dim_customers;")

invalid_customers = silver_df.join(
    customers_dim,
    on="customer_id",
    how="left_anti"
)

if spark.catalog.tableExists("fmcg.silver.invalid_customers_orders"):
    previous_table = DeltaTable.forName(spark,"fmcg.silver.invalid_customers_orders")
    previous_table.alias("target").merge(
        invalid_customers.alias("source"),
        "target.order_id = source.order_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    invalid_customers.write.format("delta")\
        .mode("overwrite")\
            .option("mergeSchema","true")\
                .option("delta.enableChangeDataFeed","true")\
                    .saveAsTable("fmcg.silver.invalid_customers_orders")


In [0]:
silver_df = silver_df.join(products , 
               on="product_id",
               how = "inner"
               )

silver_df = silver_df.join(customers_dim , 
               on="customer_id",
               how = "inner"
               )



In [0]:
silver_df = silver_df.select("order_id","product_id","product_code","customer_id","order_placement_date","order_qty","ingest_timestamp","file_name","file_size")




In [0]:
if spark.catalog.tableExists("fmcg.silver.orders"):
    old_table = DeltaTable.forName(spark,"fmcg.silver.orders")
    old_table.alias("target").merge(
        silver_df.alias("source"),
        "target.order_id = source.order_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    silver_df.write.format("delta").mode("overwrite")\
        .option("mergeSchema", "true")\
            .option("delta.enableChangeDataFeed", "true")\
                .option("overwriteSchema", "true")\
                    .saveAsTable("fmcg.silver.orders")